In [ ]:
# 06 Comparative analysis: A vs B vs C
#
# Question. How do B and C differ from A on the same validation images?
# Status. EXPLORATORY. The validation partition also selected every checkpoint (and B's grid point), so these numbers are selection-biased. The final comparison partition, the seed design and any official-test evaluation are pending the protocol decision; nothing here names a final winner.
# Runs are chosen explicitly: RUN_IDS, or B from its stage-1 selection record. The notebook never picks the latest or an arbitrary run, and stops on incomplete, failed, altered or incompatible runs.
# Cell labels: [SAFE] | [READS RUN]. No cell reads the official test set or the calibration partition.
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import json

import matplotlib.pyplot as plt
import pandas as pd

from ct_iqa.evaluation.bootstrap import paired_bootstrap_difference
from ct_iqa.experiment import report
from ct_iqa.experiment.grid import B_EXPERIMENT_ID
from ct_iqa.experiment.runs import check_compatible, read_predictions, runs_table


def show(title, obj):
    """Print a section title, then the table as plain text."""
    print(f"\n=== {title} ===")
    print(obj.to_string())   # plain text table, no interactive viewer



In [ ]:
# 1. Choose the runs [SAFE]
# Set each RUN_ID to a completed run directory name (notebooks 03-05 list them). B_FROM_SELECTION_RECORD = True takes B's run from selection_record_stage1.json instead; it is an error if that record selected no point.
IDS = {"A": "A_reference_resnet50", "B": B_EXPERIMENT_ID, "C": "C_uncertainty_aware_resnet50"}
RUN_IDS = {"A": None, "B": None, "C": None}
B_FROM_SELECTION_RECORD = False

if B_FROM_SELECTION_RECORD:
    rec_path = Path("experiments/study_v2") / B_EXPERIMENT_ID / "selection_record_stage1.json"
    rec = json.loads(rec_path.read_text())
    if not rec.get("selected"):
        raise RuntimeError(f"{rec_path.as_posix()} selected no B configuration: {rec['outcome']}")
    RUN_IDS["B"] = rec["selected"]["run_id"]
    print(f"B from {rec_path.as_posix()}: {RUN_IDS['B']} ({rec['outcome']})")

chosen = {k: Path("experiments/study_v2") / IDS[k] / "runs" / v for k, v in RUN_IDS.items() if v}
missing = [k for k, v in RUN_IDS.items() if not v]
print("chosen:", {k: v.name for k, v in chosen.items()}, "| not chosen (PENDING):", missing)


In [ ]:
# 2. Verify compatibility [READS RUN]
# Each chosen run must be completed with intact artifacts, and all must share the frozen split (SHA-256), bn_eps, input size, selection rule and the identical validation images in the same order.
S = check_compatible(chosen) if len(chosen) >= 1 else {}
if S:
    ident = {k: {"run_id": s["manifest"]["run_id"], "loss": s["manifest"]["loss"], "seed": s["manifest"]["seed"],
                 "status": s["manifest"]["status"], "selected epoch": s["manifest"]["best_epoch_1based"],
                 "device": s["manifest"]["environment"].get("device_requested"),
                 "overrides": s["manifest"].get("overrides", "-"),
                 "split SHA-256": s["manifest"]["split_manifest_sha256"][:12] + "..."} for k, s in S.items()}
    show("Runs compared (all compatible)", pd.DataFrame(ident).T)


In [ ]:
# 3. Point metrics [READS RUN]
# All six metrics on the raw 0-4 scale, n = 100 validation images, selection-biased.
if S:
    table = pd.DataFrame({k: {m: s["metrics"][m] for m in report.POINT_METRICS} for k, s in S.items()})
    show("Validation metrics at each run's selected epoch (exploratory, selection-biased)", table.round(4))


In [ ]:
# 4. Paired bootstrap differences vs A [READS RUN]
# Images are resampled jointly for both models (2000 resamples, seed 0). The interval reflects finite-sample uncertainty over these 100 images for these two fixed trained models, NOT training randomness; it is optimistic because images from one patient may be correlated. Overlap or non-overlap of intervals is not a significance test, and no significance is claimed here. Positive PLCC/SROCC/KROCC differences favour the other model; positive MSE/MAE/RMSE differences favour A.
preds = {k: read_predictions(s["run_dir"]) for k, s in S.items()}


def compare(other):
    if "A" not in preds or other not in preds:
        print(f"{other} vs A: PENDING (choose both runs)")
        return
    a, b = preds["A"], preds[other]
    res = paired_bootstrap_difference(a["label_0_4"], a["prediction_0_4"], b["prediction_0_4"], n_boot=2000, seed=0)
    show(f"{other} - A on validation (95% percentile interval, image-level bootstrap, n = {len(a['label_0_4'])})",
         pd.DataFrame(res).T[["difference_b_minus_a", "ci_low", "ci_high", "fraction_positive", "n_boot_used"]].round(4))
    report.plot_bootstrap_differences(res, f"{other} - A (validation, exploratory)")
    plt.show()


compare("B")
compare("C")


In [ ]:
# 5. Across-seed variability [READS RUN]
# Only when several completed seeds of the SAME configuration are listed. Mean and SD across seeds describe training randomness; they are reported separately from, and never pooled with, the bootstrap intervals above. Additional seeds require the pending protocol decision.
SEED_RUN_IDS = {"A": [], "B": [], "C": []}

for k, ids in SEED_RUN_IDS.items():
    if len(ids) < 2:
        print(f"{k}: fewer than 2 seed runs listed - across-seed variability not available")
        continue
    Sk = check_compatible({i: Path("experiments/study_v2") / IDS[k] / "runs" / i for i in ids})
    configs = {json.dumps({**s["config"], "train": {**s["config"]["train"], "seed": None, "device": None}}, sort_keys=True) for s in Sk.values()}
    if len(configs) != 1:
        raise RuntimeError(f"{k}: the listed runs differ in more than seed/device; they are not seed repeats")
    show(f"{k}: across-seed summary (seeds {[s['manifest']['seed'] for s in Sk.values()]})", report.seed_aggregate([s["metrics"] for s in Sk.values()]).round(4))

# Limitations
# Exploratory validation comparison only: selection-biased, one seed per arm unless listed above, image-level bootstrap without patient grouping, 100 images. A final A/B/C statement needs the approved protocol.
